# Indian Stock Trading Agent

This notebook implements an **entry/exit signal agent** for Indian (NSE/BSE) stocks using three technical indicators:

| Indicator | Entry (BUY) | Exit (SELL) |
|-----------|-------------|-------------|
| **Moving Averages** | SMA 20 crosses above SMA 50 | SMA 20 crosses below SMA 50 |
| **RSI (14)** | RSI rises above 30 (oversold recovery) | RSI falls below 70 (overbought reversal) |
| **MACD** | MACD line crosses above signal line | MACD line crosses below signal line |

A **BUY** or **SELL** signal is raised when **at least 2 of 3** conditions are met simultaneously.

### Supported Exchanges
- **NSE** - append `.NS` to the ticker (e.g. `RELIANCE.NS`, `TCS.NS`, `HDFCBANK.NS`)
- **BSE** - append `.BO` to the ticker (e.g. `RELIANCE.BO`)

In [ ]:
# Install dependencies (run once)
# !pip install yfinance pandas numpy matplotlib ta

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True

In [ ]:
# ── Configuration ────────────────────────────────────────────────
SYMBOL           = 'RELIANCE.NS'  # Change to any NSE/BSE ticker
PERIOD           = '6mo'          # 1mo | 3mo | 6mo | 1y | 2y | 5y
INITIAL_CAPITAL  = 100_000        # Starting capital in INR

In [ ]:
# ── Technical Indicator Functions ────────────────────────────────

def compute_sma(series, window):
    return series.rolling(window=window).mean()

def compute_ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast    = compute_ema(series, fast)
    ema_slow    = compute_ema(series, slow)
    macd_line   = ema_fast - ema_slow
    signal_line = compute_ema(macd_line, signal)
    histogram   = macd_line - signal_line
    return macd_line, signal_line, histogram

def compute_bollinger_bands(series, window=20, num_std=2.0):
    sma   = compute_sma(series, window)
    std   = series.rolling(window=window).std()
    upper = sma + num_std * std
    lower = sma - num_std * std
    return upper, sma, lower

print('Indicator functions defined.')

In [ ]:
# ── Signal Generation ─────────────────────────────────────────────

def generate_signals(df):
    close = df['Close'].squeeze()
    df['SMA_20'] = compute_sma(close, 20)
    df['SMA_50'] = compute_sma(close, 50)
    df['RSI']    = compute_rsi(close, 14)
    df['MACD'], df['MACD_Signal'], df['MACD_Hist'] = compute_macd(close)
    df['BB_Upper'], df['BB_Mid'], df['BB_Lower'] = compute_bollinger_bands(close)

    df['MA_Bull']   = (df['SMA_20'] > df['SMA_50']) & (df['SMA_20'].shift(1) <= df['SMA_50'].shift(1))
    df['MA_Bear']   = (df['SMA_20'] < df['SMA_50']) & (df['SMA_20'].shift(1) >= df['SMA_50'].shift(1))
    df['RSI_Bull']  = (df['RSI'] > 30) & (df['RSI'].shift(1) <= 30)
    df['RSI_Bear']  = (df['RSI'] < 70) & (df['RSI'].shift(1) >= 70)
    df['MACD_Bull'] = (df['MACD'] > df['MACD_Signal']) & (df['MACD'].shift(1) <= df['MACD_Signal'].shift(1))
    df['MACD_Bear'] = (df['MACD'] < df['MACD_Signal']) & (df['MACD'].shift(1) >= df['MACD_Signal'].shift(1))

    bull = df['MA_Bull'].astype(int) + df['RSI_Bull'].astype(int) + df['MACD_Bull'].astype(int)
    bear = df['MA_Bear'].astype(int) + df['RSI_Bear'].astype(int) + df['MACD_Bear'].astype(int)

    df['Signal'] = 'HOLD'
    df.loc[bull >= 2, 'Signal'] = 'BUY'
    df.loc[bear >= 2, 'Signal'] = 'SELL'
    return df

print('Signal generation function defined.')

In [ ]:
# ── Backtesting ───────────────────────────────────────────────────

def backtest(df, initial_capital=100_000.0):
    capital     = initial_capital
    position    = 0
    entry_price = 0.0
    trades      = []
    close       = df['Close'].squeeze()

    for date, row in df.iterrows():
        price  = float(close.loc[date])
        signal = row['Signal']

        if signal == 'BUY' and position == 0:
            shares = int(capital // price)
            if shares > 0:
                position    = shares
                entry_price = price
                capital    -= shares * price
                trades.append({'Date': date, 'Action': 'BUY',  'Price': round(price, 2), 'Shares': shares, 'Capital': round(capital, 2)})

        elif signal == 'SELL' and position > 0:
            capital += position * price
            pnl      = (price - entry_price) * position
            trades.append({'Date': date, 'Action': 'SELL', 'Price': round(price, 2), 'Shares': position, 'P&L': round(pnl, 2), 'Capital': round(capital, 2)})
            position    = 0
            entry_price = 0.0

    if position > 0:
        last_price      = float(close.iloc[-1])
        capital        += position * last_price
        unrealised_pnl  = (last_price - entry_price) * position
        trades.append({'Date': df.index[-1], 'Action': 'HOLD (open)', 'Price': round(last_price, 2), 'Shares': position, 'P&L (unrealised)': round(unrealised_pnl, 2), 'Capital': round(capital, 2)})

    return {
        'initial_capital':  initial_capital,
        'final_capital':    round(capital, 2),
        'total_return_pct': round(((capital - initial_capital) / initial_capital) * 100, 2),
        'trades':           trades,
    }

print('Backtest function defined.')

In [ ]:
# ── Fetch Data ────────────────────────────────────────────────────
ticker = yf.Ticker(SYMBOL)
df     = ticker.history(period=PERIOD)

if df.empty:
    raise ValueError(f'No data for {SYMBOL}. Check ticker symbol.')

df = generate_signals(df)
print(f'Loaded {len(df)} trading days for {SYMBOL}')
df[['Close','SMA_20','SMA_50','RSI','MACD','Signal']].tail()

In [ ]:
# ── Current Recommendation ────────────────────────────────────────
last      = df.iloc[-1]
close_now = float(df['Close'].squeeze().iloc[-1])
sig       = last['Signal']

icons = {'BUY': 'ENTRY (BUY)', 'SELL': 'EXIT (SELL)', 'HOLD': 'HOLD'}
print(f'{SYMBOL} as of {df.index[-1].date()}')
print(f'  Close Price : Rs.{close_now:,.2f}')
print(f'  SMA 20      : Rs.{last["SMA_20"]:,.2f}')
print(f'  SMA 50      : Rs.{last["SMA_50"]:,.2f}')
print(f'  RSI (14)    : {last["RSI"]:.1f}')
print(f'  MACD        : {last["MACD"]:.4f}')
print(f'  BB Upper    : Rs.{last["BB_Upper"]:,.2f}')
print(f'  BB Lower    : Rs.{last["BB_Lower"]:,.2f}')
print(f'\n  >>> RECOMMENDATION: {icons.get(sig, sig)}')

In [ ]:
# ── Price Chart with Entry/Exit Signals ──────────────────────────
close      = df['Close'].squeeze()
buy_dates  = df.index[df['Signal'] == 'BUY']
sell_dates = df.index[df['Signal'] == 'SELL']

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

ax = axes[0]
ax.plot(close,          label='Close',    linewidth=1.5, color='#1f77b4')
ax.plot(df['SMA_20'],   label='SMA 20',   linestyle='--', color='orange')
ax.plot(df['SMA_50'],   label='SMA 50',   linestyle='--', color='purple')
ax.plot(df['BB_Upper'], label='BB Upper', linestyle=':',  color='gray', alpha=0.7)
ax.plot(df['BB_Lower'], label='BB Lower', linestyle=':',  color='gray', alpha=0.7)
ax.fill_between(df.index, df['BB_Lower'], df['BB_Upper'], alpha=0.05, color='gray')
if len(buy_dates):
    ax.scatter(buy_dates,  close.loc[buy_dates],  marker='^', color='green', s=100, zorder=5, label='BUY')
if len(sell_dates):
    ax.scatter(sell_dates, close.loc[sell_dates], marker='v', color='red',   s=100, zorder=5, label='SELL')
ax.set_title(f'{SYMBOL} - Price, MAs & Bollinger Bands', fontsize=13)
ax.set_ylabel('Price (Rs.)')
ax.legend(loc='upper left', fontsize=8)

ax2 = axes[1]
ax2.plot(df['RSI'], color='#d62728', linewidth=1.2)
ax2.axhline(70, linestyle='--', color='red',   alpha=0.6, label='Overbought (70)')
ax2.axhline(30, linestyle='--', color='green', alpha=0.6, label='Oversold (30)')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.legend(fontsize=8)
ax2.set_title('RSI (14)')

ax3 = axes[2]
ax3.plot(df['MACD'],        label='MACD',   color='blue',   linewidth=1.2)
ax3.plot(df['MACD_Signal'], label='Signal', color='orange', linewidth=1.2)
ax3.bar(df.index, df['MACD_Hist'],
        color=['green' if v >= 0 else 'red' for v in df['MACD_Hist']],
        alpha=0.4, label='Histogram')
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_ylabel('MACD')
ax3.legend(fontsize=8)
ax3.set_title('MACD (12, 26, 9)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Backtest Results ──────────────────────────────────────────────
result = backtest(df, initial_capital=INITIAL_CAPITAL)

print(f'Backtest Summary - {SYMBOL} ({PERIOD})')
print(f'  Initial Capital : Rs.{result["initial_capital"]:,.2f}')
print(f'  Final Capital   : Rs.{result["final_capital"]:,.2f}')
print(f'  Total Return    : {result["total_return_pct"]:.2f}%')
print(f'  Total Trades    : {len(result["trades"])}')
print()

if result['trades']:
    trades_df = pd.DataFrame(result['trades']).set_index('Date')
    display(trades_df)

## Notes

- **Change `SYMBOL`** in the Configuration cell to analyse any NSE/BSE stock.
- Common NSE tickers: `TCS.NS`, `INFY.NS`, `HDFCBANK.NS`, `ICICIBANK.NS`, `WIPRO.NS`, `TATAMOTORS.NS`, `SBIN.NS`
- This agent is for **educational purposes only**. Always consult a SEBI-registered financial advisor before investing.
- CLI usage: `python indian_stock_agent.py --symbol TCS.NS --period 1y`